### 1. Setup

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings; warnings.filterwarnings("ignore")

sns.set_theme(style="darkgrid", context="notebook", palette="mako")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", None); pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

answers = {}   # final answer of each question, printed in the summary
MIN_N = 5      # minimum number of rows for a brand / group to count as 'reliable' in rankings

### 2. Load & clean the data

In [ ]:
DATA_PATH = "/content/mobile_sales_data.xlsx"   # <- optionally set e.g. "/content/mobiles.csv"

def find_dataset():
    exts = ("csv", "xlsx", "xls")
    pats = [f"*.{e}" for e in exts] + [f"*/*.{e}" for e in exts] + [f"/content/*.{e}" for e in exts] + [f"/content/*/*.{e}" for e in exts] \
         + [f"/mnt/user-data/uploads/*.{e}" for e in exts]
    hits = [p for pat in pats for p in glob.glob(pat) if os.path.isfile(p)]
    pref = [p for p in hits if any(k in os.path.basename(p).lower() for k in ("mobile", "phone", "sales"))]
    return (pref or hits or [None])[0]

if DATA_PATH is None:
    DATA_PATH = find_dataset()
if DATA_PATH is None:
    try:                                   # Google Colab: upload dialog
        from google.colab import files
        print("Choose your dataset file:"); DATA_PATH = next(iter(files.upload()))
    except ImportError:
        pass
if DATA_PATH is None or not os.path.isfile(DATA_PATH):
    raise FileNotFoundError("Dataset not found. Set DATA_PATH = 'full/path/to/your_file.csv' in this cell.")

df_raw = pd.read_excel(DATA_PATH) if DATA_PATH.lower().endswith(("xlsx", "xls")) else pd.read_csv(DATA_PATH)
print("Loaded:", DATA_PATH, "| shape:", df_raw.shape)
df_raw.head()

In [ ]:
df = df_raw.copy()
df.columns = (df.columns.astype(str).str.strip().str.lower().str.replace(r"[^a-z0-9]+", "_", regex=True).str.strip("_"))
df = df.loc[:, ~df.columns.str.startswith("unnamed")]

# Friendly renames for common variants
df = df.rename(columns={"brand": "brands", "model": "models", "colour": "colors", "color": "colors", "ram": "memory", "rom": "storage",
                        "sellingprice": "selling_price", "originalprice": "original_price"})
required = ["brands", "models", "memory", "storage", "selling_price", "original_price", "discount", "rating"]
missing = [c for c in required if c not in df.columns]
assert not missing, f"Missing expected columns: {missing}. Found: {list(df.columns)}"

# Text columns
for c in ["brands", "models", "colors", "mobile"]:
    if c in df.columns:
        df[c] = df[c].astype(str).str.strip().replace({"nan": np.nan})
df["brands"] = df["brands"].str.title()                       # SAMSUNG / Samsung / realme -> consistent labels
if "mobile" not in df.columns:
    df["mobile"] = df["brands"] + " " + df["models"].astype(str)

# Numeric columns (strip currency symbols / commas if present)
for c in ["memory", "storage", "selling_price", "original_price", "discount", "rating"]:
    df[c] = pd.to_numeric(df[c].astype(str).str.replace(r"[^0-9.\-]", "", regex=True), errors="coerce")

# Ratings of 0 mean 'not rated', not 'terrible': treat as missing
n_zero = (df["rating"] == 0).sum()
df.loc[df["rating"] == 0, "rating"] = np.nan
print("Ratings equal to 0 converted to NaN:", n_zero)

# Duplicates
n_dup = df.duplicated().sum(); df = df.drop_duplicates().reset_index(drop=True)
print("Exact duplicate rows removed:", n_dup)

# Check that Discount = Original - Selling, then derive discount %
calc = (df["original_price"] - df["selling_price"])
print("Rows where Discount != Original - Selling:", int(((calc - df["discount"]).abs() > 1).sum()))
df["discount_pct"] = np.where(df["original_price"] > 0, df["discount"] / df["original_price"] * 100, np.nan)

# Non-standard storage / memory sizes (typos or data errors)
std_storage = [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]
df["storage_standard"] = df["storage"].isin(std_storage)
print("Rows with non-standard storage size:", int((~df["storage_standard"]).sum()), "->", sorted(df.loc[~df["storage_standard"], "storage"].dropna().unique())[:15])

# 'Good Product' column (optional)
good_cols = [c for c in df.columns if "good" in c]
GOOD_IS_PROXY = False
if good_cols:
    gc = good_cols[0]
    s = df[gc].astype(str).str.strip().str.lower()
    df["good_product"] = s.isin(["1", "1.0", "yes", "y", "true", "good", "good product"]).astype(int)
    print(f"Using existing column '{gc}' as Good Product:", df["good_product"].value_counts().to_dict())
else:
    GOOD_IS_PROXY = True
    df["good_product"] = (df["rating"] >= 4.2).astype(int)
    df.loc[df["rating"].isna(), "good_product"] = np.nan
    print("No 'Good Product' column found -> proxy created: Good Product = rating >= 4.2 :", df["good_product"].value_counts().to_dict())

print("Final shape:", df.shape)
df.head()

## 3. EDA

### 3.1 Structure and summary statistics

In [ ]:
df.info()

In [ ]:
num_cols = ["memory", "storage", "selling_price", "original_price", "discount", "discount_pct", "rating"]
df[num_cols].describe().T

### 3.2 Missing values, unique counts

In [ ]:
miss = df.isna().sum(); miss = miss[miss > 0]
display(miss.to_frame("missing").assign(pct=lambda x: x.missing / len(df) * 100) if len(miss) else "No missing values")
print("Brands:", df["brands"].nunique(), "| unique mobile names:", df["mobile"].nunique(), "| rows:", len(df))
display(df["brands"].value_counts().head(15).to_frame("rows"))

### 3.3 Distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 9))
for ax, c in zip(axes.ravel(), ["selling_price", "original_price", "discount", "discount_pct", "rating", "storage"]):
    sns.histplot(df[c].dropna(), bins=30, kde=(c != "storage"), ax=ax, color="#0F766E"); ax.set_title(c); ax.set_xlabel("")
plt.suptitle("Distributions of key numeric variables", y=1.01, fontsize=14); plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
sns.countplot(x=df["memory"], ax=axes[0], color="#6C4AB6"); axes[0].set_title("Memory (GB) counts")
sns.countplot(x=df["storage"], ax=axes[1], color="#F2A900"); axes[1].set_title("Storage (GB) counts"); axes[1].tick_params(axis="x", rotation=60)
plt.tight_layout(); plt.show()

### 3.4 Outliers and brand-level price spread

In [ ]:
top_b = df["brands"].value_counts().head(10).index
plt.figure(figsize=(13, 5))
sns.boxplot(data=df[df.brands.isin(top_b)], x="brands", y="selling_price", order=df[df.brands.isin(top_b)].groupby("brands")["selling_price"].median().sort_values().index, palette="cubehelix")
plt.yscale("log"); plt.title("Selling price by brand (10 biggest brands, log scale)"); plt.xticks(rotation=30); plt.show()

q1, q3 = df["selling_price"].quantile([.25, .75]); iqr = q3 - q1
print("Price outliers (IQR rule):", int((df["selling_price"] > q3 + 1.5 * iqr).sum()), "- premium phones (iPhone, foldables), kept as genuine.")

### 3.5 Correlations

In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap="PuOr", center=0, linewidths=.5)
plt.title("Correlation matrix"); plt.tight_layout(); plt.show()

## 4. Answering the questions

### Which mobile brand has the highest number of mobile models in the dataset?

In [ ]:
models_per_brand = df.groupby("brands")["mobile"].nunique().sort_values(ascending=False)
display(models_per_brand.head(15).to_frame("unique_models").join(df["brands"].value_counts().rename("rows")))

top = models_per_brand.head(12)
plt.figure(figsize=(11, 5)); sns.barplot(x=top.index, y=top.values, palette="plasma")
plt.title("Number of unique models per brand (top 12)"); plt.ylabel("Models"); plt.xlabel(""); plt.xticks(rotation=35)
for i, v in enumerate(top.values): plt.text(i, v + .5, v, ha="center")
plt.tight_layout(); plt.show()

answers["Q1"] = f"{models_per_brand.index[0]} has the most models ({models_per_brand.iloc[0]}); next: {models_per_brand.index[1]} ({models_per_brand.iloc[1]}), {models_per_brand.index[2]} ({models_per_brand.iloc[2]})."
print("ANSWER:", answers["Q1"])

### Q2. Which are the top 10 mobile models based on their selling price?

In [ ]:
variants = (df.sort_values("selling_price", ascending=False)
              .drop_duplicates(subset=["mobile", "memory", "storage"]))
top10_price = variants.head(10)[["mobile", "memory", "storage", "selling_price", "original_price", "rating"]].reset_index(drop=True)
top10_price.index += 1
display(top10_price)

plt.figure(figsize=(11, 6))
lab = top10_price["mobile"] + " (" + top10_price["memory"].astype(int).astype(str) + "/" + top10_price["storage"].astype(int).astype(str) + " GB)"
sns.barplot(y=lab, x=top10_price["selling_price"], palette="magma"); plt.title("Top 10 models by selling price"); plt.xlabel("Selling price"); plt.ylabel("")
plt.tight_layout(); plt.show()

answers["Q2"] = f"Most expensive: {top10_price.iloc[0]['mobile']} at {top10_price.iloc[0]['selling_price']:,.0f}. Brands in top 10: {', '.join(variants.head(10)['brands'].value_counts().index)}."
print("ANSWER:", answers["Q2"])

### Q3. What is the average selling price of mobiles for each brand?

In [ ]:
avg_price = df.groupby("brands")["selling_price"].agg(avg_price="mean", median_price="median", rows="count").sort_values("avg_price", ascending=False)
display(avg_price.head(15))

top = avg_price.head(15)
plt.figure(figsize=(12, 5)); sns.barplot(x=top.index, y=top["avg_price"], palette="YlOrBr_r")
plt.title("Average selling price by brand (15 highest)"); plt.ylabel("Average price"); plt.xlabel(""); plt.xticks(rotation=40)
plt.tight_layout(); plt.show()

reliable = avg_price[avg_price.rows >= MIN_N]
answers["Q3"] = (f"Highest average price: {avg_price.index[0]} ({avg_price.avg_price.iloc[0]:,.0f}); lowest: {avg_price.index[-1]} ({avg_price.avg_price.iloc[-1]:,.0f}). "
                 f"Among brands with >= {MIN_N} rows: highest {reliable.index[0]} ({reliable.avg_price.iloc[0]:,.0f}), lowest {reliable.index[-1]} ({reliable.avg_price.iloc[-1]:,.0f}).")
print("ANSWER:", answers["Q3"])

### Q4. Which mobile brand offers the highest average discount?  

In [ ]:
disc = df.groupby("brands").agg(avg_discount=("discount", "mean"), avg_discount_pct=("discount_pct", "mean"), rows=("discount", "size"))
d_amt = disc.sort_values("avg_discount", ascending=False); d_pct = disc.sort_values("avg_discount_pct", ascending=False)
display(d_pct.head(10))

fig, ax = plt.subplots(1, 2, figsize=(17, 5))
sns.barplot(x=d_amt.head(10).index, y=d_amt.head(10)["avg_discount"], palette="winter_r", ax=ax[0]); ax[0].set_title("Average discount amount (top 10)"); ax[0].tick_params(axis="x", rotation=40)
sns.barplot(x=d_pct.head(10).index, y=d_pct.head(10)["avg_discount_pct"], palette="ocean_r", ax=ax[1]); ax[1].set_title("Average discount % (top 10)"); ax[1].tick_params(axis="x", rotation=40)
for a in ax: a.set_xlabel("")
plt.tight_layout(); plt.show()

rd = disc[disc.rows >= MIN_N]
answers["Q4"] = (f"Highest average discount amount: {d_amt.index[0]} ({d_amt.avg_discount.iloc[0]:,.0f}); highest average discount %: {d_pct.index[0]} ({d_pct.avg_discount_pct.iloc[0]:.1f}%). "
                 f"Among brands with >= {MIN_N} rows: {rd.avg_discount.idxmax()} by amount, {rd.avg_discount_pct.idxmax()} by percentage.")
print("ANSWER:", answers["Q4"])

### Q5. Which mobile brands have the highest average customer rating, and how does their average selling price compare?

In [ ]:
b = df.groupby("brands").agg(avg_rating=("rating", "mean"), avg_price=("selling_price", "mean"), rows=("rating", "count"))
b_all = b.sort_values("avg_rating", ascending=False)
b_rel = b[b.rows >= MIN_N].sort_values("avg_rating", ascending=False)
overall_price = df["selling_price"].mean()
top5 = b_rel.head(5).assign(price_vs_market_pct=lambda x: (x.avg_price / overall_price - 1) * 100)
display(top5)
print(f"Market average price: {overall_price:,.0f}")

fig, ax = plt.subplots(1, 2, figsize=(17, 6))
sns.barplot(x=b_rel.head(10).index, y=b_rel.head(10)["avg_rating"], palette="flare_r", ax=ax[0]); ax[0].set_ylim(b_rel.avg_rating.min() - .2, 5); ax[0].set_title(f"Top 10 brands by average rating (>= {MIN_N} rows)"); ax[0].tick_params(axis="x", rotation=40); ax[0].set_xlabel("")
sns.scatterplot(data=b_rel, x="avg_price", y="avg_rating", size="rows", sizes=(40, 400), ax=ax[1], legend=False)
for n, r in b_rel.iterrows(): ax[1].annotate(n, (r.avg_price, r.avg_rating), xytext=(4, 4), textcoords="offset points", fontsize=8)
ax[1].set_xscale("log"); ax[1].set_title("Brand average price vs average rating"); plt.tight_layout(); plt.show()

r_bp = b_rel["avg_price"].corr(b_rel["avg_rating"], method="spearman")
answers["Q5"] = (f"Top-rated brands (>= {MIN_N} rows): " + "; ".join(f"{n} ({r.avg_rating:.2f}, avg price {r.avg_price:,.0f})" for n, r in top5.iterrows()) +
                 f". Market average price {overall_price:,.0f}. Brand price vs rating Spearman rho = {r_bp:.2f}. Without the size filter the top brand is {b_all.index[0]} ({b_all.avg_rating.iloc[0]:.2f}, {int(b_all.rows.iloc[0])} rows).")
print("ANSWER:", answers["Q5"])

### Q6. How does mobile storage capacity affect the average selling price? Can you compare mobiles with different storage options?

In [ ]:
d6 = df[df["storage_standard"]]
g6 = d6.groupby("storage").agg(avg_price=("selling_price", "mean"), median_price=("selling_price", "median"),
                               avg_rating=("rating", "mean"), models=("mobile", "nunique"), rows=("storage", "size"))
g6["price_vs_prev_x"] = g6["avg_price"] / g6["avg_price"].shift()
display(g6)

fig, ax = plt.subplots(1, 2, figsize=(17, 5.5))
sns.barplot(x=g6.index.astype(int), y=g6["avg_price"], palette="crest", ax=ax[0]); ax[0].set_title("Average selling price by storage (GB)")
for i, (s, v) in enumerate(g6["avg_price"].items()): ax[0].text(i, v, f"{v:,.0f}", ha="center", va="bottom", fontsize=9)
sns.boxplot(data=d6, x="storage", y="selling_price", ax=ax[1], palette="crest"); ax[1].set_yscale("log"); ax[1].set_title("Price distribution by storage (log scale)")
plt.tight_layout(); plt.show()

rho = d6["storage"].corr(d6["selling_price"], method="spearman")
lo, hi = g6.index.min(), g6.index.max()
answers["Q6"] = (f"Price rises with storage (Spearman rho = {rho:.2f}): average price goes from {g6.avg_price.iloc[0]:,.0f} at {lo:.0f} GB to {g6.avg_price.iloc[-1]:,.0f} at {hi:.0f} GB. "
                 + "; ".join(f"{int(s)} GB: {v:,.0f}" for s, v in g6["avg_price"].items()) + ".")
print("ANSWER:", answers["Q6"])

### Q7. Which mobile models provide the highest discount amount, and what percentage of their original price is discounted?

In [ ]:
v7 = (df.sort_values("discount", ascending=False).drop_duplicates(subset=["mobile", "memory", "storage"]))
top_disc = v7.head(10)[["mobile", "memory", "storage", "original_price", "selling_price", "discount", "discount_pct"]].reset_index(drop=True)
top_disc.index += 1
display(top_disc)

fig, ax = plt.subplots(1, 2, figsize=(17, 6))
lab = top_disc["mobile"].str.slice(0, 32)
sns.barplot(y=lab, x=top_disc["discount"], palette="rocket_r", ax=ax[0]); ax[0].set_title("Top 10 by discount amount"); ax[0].set_ylabel("")
sns.barplot(y=lab, x=top_disc["discount_pct"], palette="autumn_r", ax=ax[1]); ax[1].set_title("...and their discount % of original price"); ax[1].set_ylabel("")
plt.tight_layout(); plt.show()

t = top_disc.iloc[0]
answers["Q7"] = (f"Largest discount: {t['mobile']} ({t['discount']:,.0f} off, {t['discount_pct']:.1f}% of its original price). "
                 f"The top-10 discounts range from {top_disc.discount_pct.min():.1f}% to {top_disc.discount_pct.max():.1f}% of the original price.")
print("ANSWER:", answers["Q7"])

### Q8. Which combination of memory and storage offers the best balance between selling price and customer rating?

In [ ]:
d8 = df[df["storage_standard"]].dropna(subset=["rating"])
combo = d8.groupby(["memory", "storage"]).agg(avg_price=("selling_price", "mean"), avg_rating=("rating", "mean"), rows=("rating", "size")).reset_index()
combo = combo[combo.rows >= MIN_N].copy()
def mm(s): return (s - s.min()) / (s.max() - s.min()) if s.max() > s.min() else s * 0
combo["balance_score"] = (0.5 * mm(combo["avg_rating"]) + 0.5 * (1 - mm(np.log(combo["avg_price"])))) * 100
combo["combo"] = combo["memory"].astype(int).astype(str) + " GB RAM / " + combo["storage"].astype(int).astype(str) + " GB"
combo = combo.sort_values("balance_score", ascending=False).reset_index(drop=True)
display(combo[["combo", "avg_price", "avg_rating", "rows", "balance_score"]].head(10))

fig, ax = plt.subplots(1, 2, figsize=(17, 6))
sns.scatterplot(data=combo, x="avg_price", y="avg_rating", size="rows", sizes=(50, 500), hue="balance_score", palette="plasma", ax=ax[0], legend=False)
for _, r in combo.iterrows(): ax[0].annotate(f"{int(r.memory)}/{int(r.storage)}", (r.avg_price, r.avg_rating), xytext=(4, 4), textcoords="offset points", fontsize=8)
ax[0].set_xscale("log"); ax[0].set_title("RAM/Storage combos: price vs rating")
sns.barplot(y=combo["combo"].head(10), x=combo["balance_score"].head(10), palette="plasma", ax=ax[1]); ax[1].set_title("Top 10 combos by balance score"); ax[1].set_ylabel("")
plt.tight_layout(); plt.show()

b8 = combo.iloc[0]
answers["Q8"] = (f"Best balance: {b8['combo']} (avg price {b8.avg_price:,.0f}, avg rating {b8.avg_rating:.2f}, {int(b8.rows)} rows, score {b8.balance_score:.0f}/100); "
                 f"runner-up: {combo.iloc[1]['combo']}.")
print("ANSWER:", answers["Q8"])

### Q9. Can you identify the top 10 mobile models that provide the best value for money by considering selling price, discount, storage, memory, and customer rating? Explain how you calculated the value score.

In [ ]:
v9 = df.dropna(subset=["rating", "discount_pct"]).copy()
v9 = v9.sort_values("rating", ascending=False).drop_duplicates(subset=["mobile", "memory", "storage"])
v9 = v9[v9["selling_price"] > 0]
comp = pd.DataFrame({
    "rating_s":   mm(v9["rating"]),
    "price_s":    1 - mm(np.log(v9["selling_price"])),
    "discount_s": mm(v9["discount_pct"]),
    "memory_s":   mm(v9["memory"]),
    "storage_s":  mm(np.log(v9["storage"].clip(lower=1))),
})
w9 = {"rating_s": .30, "price_s": .25, "discount_s": .15, "memory_s": .15, "storage_s": .15}
v9["value_score"] = sum(comp[k] * w for k, w in w9.items()) * 100
best9 = v9.nlargest(10, "value_score")[["mobile", "memory", "storage", "selling_price", "discount_pct", "rating", "value_score"]].reset_index(drop=True)
best9.index += 1
display(best9)

plt.figure(figsize=(11, 6))
lab = best9["mobile"].str.slice(0, 35) + " (" + best9["memory"].astype(int).astype(str) + "/" + best9["storage"].astype(int).astype(str) + ")"
sns.barplot(y=lab, x=best9["value_score"], palette="ocean_r"); plt.title("Top 10 value-for-money models"); plt.xlabel("Value score"); plt.ylabel("")
plt.tight_layout(); plt.show()

# Robustness: equal weights
eq = comp.mean(axis=1)
overlap = len(set(v9.loc[eq.nlargest(10).index, "mobile"]) & set(best9["mobile"]))
print(f"Overlap of top 10 with an equal-weights version of the score: {overlap}/10")

answers["Q9"] = (f"Best value: {best9.iloc[0]['mobile']} (score {best9.iloc[0]['value_score']:.0f}; price {best9.iloc[0]['selling_price']:,.0f}, rating {best9.iloc[0]['rating']}). "
                 f"Next: {', '.join(best9['mobile'].iloc[1:4])}. Equal-weights overlap: {overlap}/10.")
print("ANSWER:", answers["Q9"])

### Q10.  Can you analyze the factors that determine whether a mobile is classified as a "Good Product"? Which factors such as price, discount, rating, memory, or storage appear to have the strongest relationship with Good Product?

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

feats = ["selling_price", "discount", "discount_pct", "rating", "memory", "storage"]
if GOOD_IS_PROXY:
    feats.remove("rating")
    print("'Good Product' is a proxy defined from rating, so rating is excluded from the predictors to avoid circular logic.")
d10 = df.dropna(subset=feats + ["good_product"]).copy()
y = d10["good_product"].astype(int)
print("Good Product share:", f"{y.mean():.1%}", "| rows:", len(d10))
assert y.nunique() == 2, "Good Product has a single class, so there is nothing to compare."

# Group means
display(d10.groupby("good_product")[feats].mean().T.rename(columns={0: "Not good", 1: "Good"}))

# (1) correlation (point-biserial)
corr_gp = d10[feats].corrwith(y)

# (2) standardized logistic regression
X = StandardScaler().fit_transform(d10[feats])
lr = LogisticRegression(max_iter=2000).fit(X, y)
coef = pd.Series(lr.coef_[0], index=feats)

# (3) random forest importance + held-out AUC
Xtr, Xte, ytr, yte = train_test_split(d10[feats], y, test_size=.25, random_state=42, stratify=y)
rf = RandomForestClassifier(n_estimators=300, random_state=42, min_samples_leaf=3).fit(Xtr, ytr)
imp = pd.Series(rf.feature_importances_, index=feats)
print(f"Random forest hold-out AUC: {roc_auc_score(yte, rf.predict_proba(Xte)[:, 1]):.3f}")

res = pd.DataFrame({"correlation": corr_gp, "logit_coef_std": coef, "rf_importance": imp})
res["avg_rank"] = pd.concat([res.correlation.abs().rank(ascending=False), res.logit_coef_std.abs().rank(ascending=False), res.rf_importance.rank(ascending=False)], axis=1).mean(axis=1)
res = res.sort_values("avg_rank"); display(res)

fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))
corr_gp.sort_values().plot.barh(ax=ax[0], color="#0F766E"); ax[0].set_title("Correlation with Good Product")
coef.sort_values().plot.barh(ax=ax[1], color="#E4572E"); ax[1].set_title("Logistic coefficients (standardized)")
imp.sort_values().plot.barh(ax=ax[2], color="#17BEBB"); ax[2].set_title("Random forest importance")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, len(feats), figsize=(4 * len(feats), 4))
for a, c in zip(np.atleast_1d(axes), feats):
    sns.boxplot(data=d10, x="good_product", y=c, ax=a, palette="cubehelix", showfliers=False); a.set_title(c); a.set_xlabel("Good Product (0/1)")
plt.tight_layout(); plt.show()

strongest = res.index[0]
answers["Q10"] = (f"Strongest factor: {strongest} (corr {res.loc[strongest,'correlation']:+.2f}). Ranking: " +
                  " > ".join(f"{n} ({r.correlation:+.2f})" for n, r in res.iterrows()) +
                  (". Note: Good Product was a rating-based proxy, so rating was excluded." if GOOD_IS_PROXY else "."))
print("ANSWER:", answers["Q10"])